# LITE Reproduction: First 64 UCR Datasets

Reproduces **LITE** (Ismail-Fawaz et al., *LITE: Light Inception with boosTing tEchniques for Time Series Classification*, DSAA 2023) on the **first 64 UCR univariate datasets in alphabetical order** (ACSF1 to InsectWingbeatSound), using the authors' code from [MSD-IRIMAS/LITE](https://github.com/MSD-IRIMAS/LITE).

**Protocol** (as in the paper): 5 runs per dataset, 1500 epochs, batch size 64, Adam, learning rate halved on training-loss plateaus, best model chosen by training loss. `LITE-mean` / `LITE-std` are over the 5 runs; `LITETime` is the ensemble of the 5 models.

**Notes**
- Source pinned to commit `e62f643`. That commit ships with `n_epochs=2`; it is set back to the paper's 1500.
- TensorFlow 2.16.1 needs Python ≤ 3.12, so training runs in a separate Python 3.11 environment.
- UCR data is downloaded automatically by `aeon`.
- Run with emissions tracking off (`--track-emissions False`).

**How to run on Kaggle:** Settings → *Internet On*, *Accelerator GPU T4 x2*, then *Run All* (or *Save Version → Save & Run All*). Nothing needs to be uploaded or attached. A Kaggle session stops after 12 hours; datasets already in `results_ucr.csv` are skipped, so re-running continues where it stopped as long as `/kaggle/working` is kept (e.g. from the previous version's output).

**Output:** `/kaggle/working/results_ucr.csv` (columns `dataset, LITE-mean, LITE-std, LITETime`), plus checkpoints and plots in `/kaggle/working/lite_benchmark/LITE/run_<r>/<dataset>/`.

In [ ]:
# 1. Paths
import os, glob, json, time, subprocess
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working")
REPO = WORK / "LITE"
OUTPUT_PARENT = WORK / "lite_benchmark"     # main.py writes to OUTPUT_PARENT/LITE/
RESULTS = OUTPUT_PARENT / "LITE"
LOGS = WORK / "logs"
for p in (RESULTS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
N_RUNS = 5

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])

In [ ]:
# 2.
# Original LITE code, pinned to the version used in this study
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/MSD-IRIMAS/LITE.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "e62f643"], check=True)

# This commit ships with n_epochs=2 (a debugging default); the paper uses 1500
lite_py = REPO / "src/classifiers/lite.py"
code_txt = lite_py.read_text()
if "n_epochs=1500," not in code_txt:
    assert code_txt.count("n_epochs=2,") == 1, "lite.py is not the expected version"
    lite_py.write_text(code_txt.replace("n_epochs=2,", "n_epochs=1500,"))
print([l.strip() for l in lite_py.read_text().splitlines() if "n_epochs=" in l])   # expect ['n_epochs=1500,']

In [ ]:
# 3.
# Python 3.11 environment with the package versions used in this study (about 3 minutes)
!pip install -q uv
!uv python install 3.11
!uv venv /kaggle/working/lite_env311 --python 3.11 --allow-existing
!UV_LINK_MODE=copy uv pip install -q --python /kaggle/working/lite_env311/bin/python \
    "tensorflow[and-cuda]==2.16.1" "keras==3.6.0" "numpy==1.26.4" "scikit-learn==1.4.2" \
    "pandas==2.0.3" "matplotlib==3.9.0" "codecarbon" "aeon==1.3.0"

In [ ]:
# 4.
# Environment for the Python 3.11 subprocesses, and GPU check
PY = "/kaggle/working/lite_env311/bin/python"
SITE = "/kaggle/working/lite_env311/lib/python3.11/site-packages"
ENV = os.environ.copy()
ENV["LD_LIBRARY_PATH"] = ":".join(sorted(glob.glob(f"{SITE}/nvidia/*/lib")) + [ENV.get("LD_LIBRARY_PATH", "")])
ENV["PATH"] = f"{SITE}/nvidia/cuda_nvcc/bin:" + ENV["PATH"]
ENV["MPLBACKEND"] = "Agg"            # Kaggle's inline plotting backend is not available in the venv
ENV["TF_CPP_MIN_LOG_LEVEL"] = "2"
ENV.pop("PYTHONPATH", None)          # keep the kernel's Python 3.13 packages out

check = ("import tensorflow as tf, keras, numpy, aeon; g = tf.config.list_physical_devices('GPU'); "
         "print('TF', tf.__version__, '| Keras', keras.__version__, '| NumPy', numpy.__version__, "
         "'| aeon', aeon.__version__, '| GPUs:', len(g))")
r = subprocess.run([PY, "-c", check], env=ENV, capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])
N_GPUS = int(r.stdout.split("GPUs:")[-1]) if "GPUs:" in r.stdout else 0
assert N_GPUS >= 1, "No GPU found: Settings -> Accelerator -> GPU T4 x2"

In [ ]:
# 5. The first 64 UCR datasets in alphabetical order
r = subprocess.run([PY, "-c", "from aeon.datasets.tsc_datasets import univariate; print('\\n'.join(sorted(univariate)))"],
                   env=ENV, capture_output=True, text=True, check=True)
UCR128 = r.stdout.split()
assert len(UCR128) == 128, len(UCR128)
DATASETS = UCR128[:64]
print(len(DATASETS), "datasets:", DATASETS[0], "...", DATASETS[-1])

In [ ]:
# 6. Train (one dataset at a time, GPU 0). Datasets already in results_ucr.csv are skipped.
#    Per-epoch training output goes to /kaggle/working/logs/<dataset>.log
csv_path = RESULTS / "results_ucr.csv"
done = set(pd.read_csv(csv_path)["dataset"]) if csv_path.exists() else set()
todo = [d for d in DATASETS if d not in done]
print(f"{len(DATASETS) - len(todo)} already done, {len(todo)} to train", flush=True)

train_env = dict(ENV, CUDA_VISIBLE_DEVICES="0")
for i, ds in enumerate(todo, 1):
    t0 = time.time()
    with open(LOGS / f"{ds}.log", "a") as f:
        rc = subprocess.run(
            [PY, "main.py", "--dataset", ds, "--classifier", "LITE", "--runs", str(N_RUNS),
             "--output-directory", str(OUTPUT_PARENT) + "/",      # trailing slash required by main.py
             "--track-emissions", "False"],
            cwd=REPO, env=train_env, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print((LOGS / f"{ds}.log").read_text()[-3000:])
        raise RuntimeError(f"{ds} failed - see {LOGS / f'{ds}.log'}")
    row = pd.read_csv(csv_path).set_index("dataset").loc[ds]
    print(f"[{i}/{len(todo)}] {ds:28} {(time.time()-t0)/60:6.1f} min | "
          f"LITE-mean {row['LITE-mean']:.4f} | LITETime {row['LITETime']:.4f}", flush=True)
print("Training finished.")

In [ ]:
# 7. Results: save locally and compare with the paper's per-dataset results
from io import StringIO
from scipy.stats import wilcoxon

res = pd.read_csv(RESULTS / "results_ucr.csv")
res["order"] = res.dataset.map({d: i for i, d in enumerate(DATASETS)})
res = res.sort_values("order").drop(columns="order")
res.to_csv(WORK / "results_ucr.csv", index=False)
print("Saved", WORK / "results_ucr.csv", "with", len(res), "datasets")

# The authors' per-dataset results were in their repository before commit ba43d96
paper = pd.read_csv(StringIO(subprocess.run(["git", "-C", str(REPO), "show", "ba43d96^:results.csv"],
                                            capture_output=True, text=True, check=True).stdout)).set_index("dataset")
cmp = res.set_index("dataset").join(paper[["LITE", "LITETime"]].add_prefix("paper_"), how="inner")
cmp["diff_LITE"] = cmp["LITE-mean"] - cmp["paper_LITE"]
cmp["diff_LITETime"] = cmp["LITETime"] - cmp["paper_LITETime"]
cmp.to_csv(WORK / "reproduction_vs_paper.csv")

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 100)
display(cmp)
for ours, theirs, d in [("LITE-mean", "paper_LITE", "diff_LITE"), ("LITETime", "paper_LITETime", "diff_LITETime")]:
    x = cmp[d]
    print(f"{ours:10} ours {cmp[ours].mean():.4f} | paper {cmp[theirs].mean():.4f} | diff {x.mean():+.4f} | "
          f"better/same/worse {(x > 1e-9).sum()}/{(x.abs() <= 1e-9).sum()}/{(x < -1e-9).sum()} | "
          f"within 1 point {(x.abs() <= 0.01).sum()}/{len(x)} | Wilcoxon p {wilcoxon(x[x.abs() > 1e-9]).pvalue:.3f}")